In [1]:
"""
Stratified random sampling for manual validation (n=68)
- Reads Stage 3 run-level dataset (run_metrics_v16_stage3_enhanced.csv) from:
    C:\Android Mobile App\ICST2026_Ext\0-Data_Feb15
- Produces a sample list for manual review saved to:
    C:\Android Mobile App\ICST2026_Ext\0-Sampling

Sampling plan (default):
  - 24 GMD-labeled runs
  - 24 Custom-labeled runs (Emu_Custom)
  - 20 Other styles
Within each stratum:
  - ~60% TTFTS-present
  - ~40% TTFTS-absent
Plus:
  - per-repo cap (default max 5 samples per repo)
  - reproducible seed (default 20260215)

Outputs:
  - manual_sample_runs_n68.csv
  - manual_sample_runs_n68_run_ids.txt
"""

from __future__ import annotations

import os
import re
import math
import json
import random
from datetime import datetime
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd


# -------------------------
# USER CONFIG
# -------------------------
DATA_DIR = r"C:\Android Mobile App\ICST2026_Ext\0-Data_Feb15"
OUT_DIR  = r"C:\Android Mobile App\ICST2026_Ext\0-Sampling"

RUN_METRICS_FILE = "run_metrics_v16_stage3_enhanced.csv"  # expected inside DATA_DIR

SEED = 20260215
TOTAL_N = 68

# Stratum sizes (must sum to TOTAL_N)
N_GMD = 24
N_CUSTOM = 24
N_OTHER = 20

# TTFTS presence ratio per stratum
TTFTS_PRESENT_RATIO = 0.60  # ~60% present, 40% absent

# Max samples per repo to avoid dominance
MAX_PER_REPO = 5

# If your "Custom" style label is different, change here:
CUSTOM_STYLE_TOKEN = "Emu_Custom"
GMD_STYLE_TOKEN = "GMD"


# -------------------------
# HELPERS
# -------------------------

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def pick_first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_style_str(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    # unify separators
    s = s.replace(";", ",").replace("|", ",")
    return s


def has_style(style_field: str, token: str) -> bool:
    s = normalize_style_str(style_field).lower()
    t = token.lower()
    # match whole-token-ish (commas/spaces)
    # examples: "GMD", "GMD, Emu_Custom", "['GMD']"
    if t in s:
        # avoid accidental substring matches (rare) by boundary check
        return bool(re.search(rf"(^|[^a-z0-9]){re.escape(t)}([^a-z0-9]|$)", s))
    return False


def compute_ttfts_present(df: pd.DataFrame) -> pd.Series:
    # Prefer explicit numeric field
    ttfts_col = pick_first_existing_col(df, ["ttfts_seconds", "ttfts_sec", "ttfts"])
    if ttfts_col is not None:
        return pd.to_numeric(df[ttfts_col], errors="coerce").notna()

    # Fallback: presence of step timestamps flag
    step_flag_col = pick_first_existing_col(df, ["has_step_timestamps", "has_steps_timestamps", "steps_available"])
    if step_flag_col is not None:
        return df[step_flag_col].astype(str).str.lower().isin(["1", "true", "yes", "y"])

    # Last resort: try job/step timing columns if exist
    any_time_cols = [c for c in df.columns if "step" in c.lower() and ("start" in c.lower() or "started" in c.lower())]
    if any_time_cols:
        return df[any_time_cols].notna().any(axis=1)

    return pd.Series([False] * len(df), index=df.index)


def balanced_sample(
    df: pd.DataFrame,
    n_total: int,
    ttfts_present_ratio: float,
    repo_col: str,
    rng: np.random.Generator,
    max_per_repo: int
) -> pd.DataFrame:
    """
    Sample n_total rows from df, aiming for given ttfts_present_ratio.
    Enforces max_per_repo.
    """
    if n_total <= 0:
        return df.head(0).copy()

    df = df.copy()
    if df.empty:
        return df

    df["_ttfts_present"] = compute_ttfts_present(df)

    n_present = int(round(n_total * ttfts_present_ratio))
    n_absent = n_total - n_present

    present_df = df[df["_ttfts_present"]].copy()
    absent_df  = df[~df["_ttfts_present"]].copy()

    # If not enough present/absent, rebalance
    if len(present_df) < n_present:
        n_present = len(present_df)
        n_absent = n_total - n_present
    if len(absent_df) < n_absent:
        n_absent = len(absent_df)
        n_present = n_total - n_absent

    def sample_with_repo_cap(pool: pd.DataFrame, k: int) -> pd.DataFrame:
        if k <= 0 or pool.empty:
            return pool.head(0).copy()

        # Shuffle deterministically
        pool = pool.sample(frac=1.0, random_state=int(rng.integers(0, 2**31 - 1))).copy()

        picked_rows = []
        repo_counts: Dict[str, int] = {}

        for _, row in pool.iterrows():
            repo = str(row[repo_col])
            repo_counts.setdefault(repo, 0)
            if repo_counts[repo] >= max_per_repo:
                continue
            picked_rows.append(row)
            repo_counts[repo] += 1
            if len(picked_rows) >= k:
                break

        if len(picked_rows) < k:
            # If cap prevents reaching k, relax cap slightly for this pool only
            # (still deterministic order)
            for _, row in pool.iterrows():
                if len(picked_rows) >= k:
                    break
                if row.to_dict() in [r.to_dict() for r in picked_rows]:
                    continue
                picked_rows.append(row)

        return pd.DataFrame(picked_rows)

    s_present = sample_with_repo_cap(present_df, n_present)
    s_absent  = sample_with_repo_cap(absent_df,  n_absent)

    out = pd.concat([s_present, s_absent], ignore_index=True)

    # If still short, fill from remaining regardless of ttfts status
    if len(out) < n_total:
        used_ids = set(out.index)
        remaining = df.drop(columns=["_ttfts_present"]).copy()
        remaining = remaining.sample(frac=1.0, random_state=int(rng.integers(0, 2**31 - 1)))
        # enforce repo cap cumulatively
        repo_counts = out[repo_col].astype(str).value_counts().to_dict()
        extra = []
        for _, row in remaining.iterrows():
            repo = str(row[repo_col])
            repo_counts.setdefault(repo, 0)
            if repo_counts[repo] >= max_per_repo:
                continue
            extra.append(row)
            repo_counts[repo] += 1
            if len(out) + len(extra) >= n_total:
                break
        if extra:
            out = pd.concat([out, pd.DataFrame(extra)], ignore_index=True)

    return out.drop(columns=["_ttfts_present"], errors="ignore")


# -------------------------
# MAIN
# -------------------------

def main() -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    rng = np.random.default_rng(SEED)

    ensure_dir(OUT_DIR)

    run_metrics_path = os.path.join(DATA_DIR, RUN_METRICS_FILE)
    if not os.path.exists(run_metrics_path):
        raise FileNotFoundError(f"Could not find: {run_metrics_path}")

    df = pd.read_csv(run_metrics_path, low_memory=False)

    # Identify key columns robustly
    run_id_col = pick_first_existing_col(df, ["run_id", "id", "workflow_run_id"])
    repo_col   = pick_first_existing_col(df, ["repo_full_name", "repo", "repository", "full_name"])
    style_col  = pick_first_existing_col(df, ["styles", "style", "execution_style", "exec_style", "style_label"])

    if run_id_col is None:
        raise ValueError("Could not find a run id column (tried: run_id/id/workflow_run_id).")
    if repo_col is None:
        raise ValueError("Could not find a repo column (tried: repo_full_name/repo/repository/full_name).")
    if style_col is None:
        raise ValueError("Could not find a styles column (tried: styles/style/execution_style/exec_style/style_label).")

    # Build strata
    df["_is_gmd"] = df[style_col].apply(lambda x: has_style(x, GMD_STYLE_TOKEN))
    df["_is_custom"] = df[style_col].apply(lambda x: has_style(x, CUSTOM_STYLE_TOKEN))

    gmd_df = df[df["_is_gmd"]].copy()
    custom_df = df[(~df["_is_gmd"]) & (df["_is_custom"])].copy()  # keep GMD separate if mixed
    other_df = df[(~df["_is_gmd"]) & (~df["_is_custom"])].copy()

    # Sample each stratum
    s_gmd = balanced_sample(gmd_df, N_GMD, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
    s_custom = balanced_sample(custom_df, N_CUSTOM, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
    s_other = balanced_sample(other_df, N_OTHER, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)

    sample_df = pd.concat([s_gmd, s_custom, s_other], ignore_index=True)

    # Ensure uniqueness by run_id; if duplicates, top up from same strata
    sample_df = sample_df.drop_duplicates(subset=[run_id_col]).copy()

    def top_up_from(pool: pd.DataFrame, needed: int, already: pd.DataFrame) -> pd.DataFrame:
        if needed <= 0:
            return already
        existing_ids = set(already[run_id_col].astype(str))
        pool2 = pool[~pool[run_id_col].astype(str).isin(existing_ids)].copy()
        add = balanced_sample(pool2, needed, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
        out = pd.concat([already, add], ignore_index=True).drop_duplicates(subset=[run_id_col])
        return out

    if len(sample_df) < TOTAL_N:
        missing = TOTAL_N - len(sample_df)
        # Prefer topping up from the "other" stratum first (least constrained), then custom, then gmd
        sample_df = top_up_from(other_df, missing, sample_df)
        missing = TOTAL_N - len(sample_df)
        if missing > 0:
            sample_df = top_up_from(custom_df, missing, sample_df)
        missing = TOTAL_N - len(sample_df)
        if missing > 0:
            sample_df = top_up_from(gmd_df, missing, sample_df)

    # Final trim if we overshot
    if len(sample_df) > TOTAL_N:
        sample_df = sample_df.sample(n=TOTAL_N, random_state=SEED).copy()

    # Add convenience columns for reviewers
    sample_df["_sample_seed"] = SEED
    sample_df["_sample_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sample_df["_stratum"] = np.where(sample_df[style_col].apply(lambda x: has_style(x, GMD_STYLE_TOKEN)), "GMD",
                           np.where(sample_df[style_col].apply(lambda x: has_style(x, CUSTOM_STYLE_TOKEN)), "Custom", "Other"))
    sample_df["_ttfts_present"] = compute_ttfts_present(sample_df)

    # Keep key columns first
    cols_front = [repo_col, run_id_col, style_col, "_stratum", "_ttfts_present"]
    cols_front += [c for c in ["conclusion", "status", "event", "created_at", "run_created_at"] if c in sample_df.columns]
    remaining = [c for c in sample_df.columns if c not in cols_front]
    sample_df = sample_df[cols_front + remaining]

    # Save outputs
    out_csv = os.path.join(OUT_DIR, "manual_sample_runs_n68.csv")
    out_txt = os.path.join(OUT_DIR, "manual_sample_runs_n68_run_ids.txt")
    out_meta = os.path.join(OUT_DIR, "manual_sample_runs_n68_meta.json")

    sample_df.to_csv(out_csv, index=False)

    with open(out_txt, "w", encoding="utf-8") as f:
        for rid in sample_df[run_id_col].astype(str).tolist():
            f.write(rid + "\n")

    meta = {
        "seed": SEED,
        "total_n": TOTAL_N,
        "n_gmd": N_GMD,
        "n_custom": N_CUSTOM,
        "n_other": N_OTHER,
        "ttfts_present_ratio_target": TTFTS_PRESENT_RATIO,
        "max_per_repo": MAX_PER_REPO,
        "source_file": run_metrics_path,
        "run_id_col": run_id_col,
        "repo_col": repo_col,
        "style_col": style_col,
        "counts_by_stratum": sample_df["_stratum"].value_counts().to_dict(),
        "counts_ttfts_present": sample_df["_ttfts_present"].value_counts().to_dict(),
    }
    with open(out_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("Saved sample to:")
    print(" -", out_csv)
    print(" -", out_txt)
    print(" -", out_meta)
    print("\nQuick summary:")
    print(sample_df["_stratum"].value_counts())
    print(sample_df["_ttfts_present"].value_counts())


if __name__ == "__main__":
    # Sanity check the target sizes add up
    if (N_GMD + N_CUSTOM + N_OTHER) != TOTAL_N:
        raise ValueError("N_GMD + N_CUSTOM + N_OTHER must equal TOTAL_N.")
    main()


Saved sample to:
 - C:\Android Mobile App\ICST2026_Ext\0-Sampling\manual_sample_runs_n68.csv
 - C:\Android Mobile App\ICST2026_Ext\0-Sampling\manual_sample_runs_n68_run_ids.txt
 - C:\Android Mobile App\ICST2026_Ext\0-Sampling\manual_sample_runs_n68_meta.json

Quick summary:
_stratum
GMD       24
Custom    24
Other     20
Name: count, dtype: int64
_ttfts_present
True     40
False    28
Name: count, dtype: int64


In [ ]:
## same stratified sampling with updated records from 3C

In [14]:
"""
Stratified random sampling (n=68) from Stage 3A, BUT final output is:
- Keep ONLY: run_id + _stratum from the sample
- Then pull ALL rows for those run_ids from Stage 3C (one row per style if multi-style)
- Final dataset = ALL Stage3C columns + one added column: _stratum (from the sample)

Outputs (in OUT_DIR):
- sampled_3c_all_rows_n68.csv
- sampled_3c_all_rows_n68_run_ids.txt
- sampled_3c_all_rows_n68_meta.json
"""

from __future__ import annotations

import os
import re
import json
import random
from datetime import datetime
from typing import Optional, List, Dict

import numpy as np
import pandas as pd


# -------------------------
# USER CONFIG
# -------------------------
DATA_DIR = r"C:\Android Mobile App\ICST2026_Ext\0-Data_Feb15"
OUT_DIR  = r"C:\Android Mobile App\ICST2026_Ext\0-Sampling_v2.0"

RUN_METRICS_FILE = "run_metrics_v16_stage3_enhanced.csv"           # Stage 3A (sampling frame)
STAGE3C_PATH     = r"C:\Android Mobile App\ICST2026_Ext\run_per_style_v1_stage3.csv"  # Stage 3C

SEED = 20260215
TOTAL_N = 68

# Stratum sizes (must sum to TOTAL_N)
N_GMD = 24
N_CUSTOM = 24
N_OTHER = 20

# TTFTS presence ratio per stratum (computed on Stage 3A ttfts_seconds)
TTFTS_PRESENT_RATIO = 0.60

# Max samples per repo
MAX_PER_REPO = 5

# Tokens used for stratum detection (from Stage 3A styles)
CUSTOM_STYLE_TOKEN = "Emu_Custom"
GMD_STYLE_TOKEN = "GMD"

OUT_CSV_NAME = "sampled_3c_all_rows_n68.csv"


# -------------------------
# HELPERS
# -------------------------
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def pick_first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_style_str(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.replace(";", ",").replace("|", ",")
    return s

def has_style(style_field: str, token: str) -> bool:
    s = normalize_style_str(style_field).lower()
    t = token.lower()
    if t in s:
        return bool(re.search(rf"(^|[^a-z0-9]){re.escape(t)}([^a-z0-9]|$)", s))
    return False

def normalize_run_id_series(s: pd.Series) -> pd.Series:
    # Handles Excel-like run_id reading where it becomes float (e.g., 123.0)
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

def compute_ttfts_present(df: pd.DataFrame) -> pd.Series:
    ttfts_col = pick_first_existing_col(df, ["ttfts_seconds", "ttfts_sec", "ttfts"])
    if ttfts_col is not None:
        return pd.to_numeric(df[ttfts_col], errors="coerce").notna()
    return pd.Series([False] * len(df), index=df.index)

def balanced_sample(
    df: pd.DataFrame,
    n_total: int,
    ttfts_present_ratio: float,
    repo_col: str,
    rng: np.random.Generator,
    max_per_repo: int,
) -> pd.DataFrame:
    """
    Picks n_total rows from df with approx ttfts_present_ratio split,
    respecting max_per_repo cap (relaxes if needed).
    """
    if n_total <= 0 or df.empty:
        return df.head(0).copy()

    df = df.copy()
    df["_ttfts_present"] = compute_ttfts_present(df)

    n_present = int(round(n_total * ttfts_present_ratio))
    n_absent = n_total - n_present

    present_df = df[df["_ttfts_present"]].copy()
    absent_df  = df[~df["_ttfts_present"]].copy()

    if len(present_df) < n_present:
        n_present = len(present_df)
        n_absent = n_total - n_present
    if len(absent_df) < n_absent:
        n_absent = len(absent_df)
        n_present = n_total - n_absent

    def sample_with_repo_cap(pool: pd.DataFrame, k: int) -> pd.DataFrame:
        if k <= 0 or pool.empty:
            return pool.head(0).copy()

        pool = pool.sample(frac=1.0, random_state=int(rng.integers(0, 2**31 - 1))).copy()
        picked = []
        repo_counts: Dict[str, int] = {}

        for _, row in pool.iterrows():
            repo = str(row[repo_col])
            repo_counts.setdefault(repo, 0)
            if repo_counts[repo] >= max_per_repo:
                continue
            picked.append(row)
            repo_counts[repo] += 1
            if len(picked) >= k:
                break

        # relax cap if still short
        if len(picked) < k:
            for _, row in pool.iterrows():
                if len(picked) >= k:
                    break
                rid = str(row.get("run_id", row.get("id", "")))
                if any(str(r.get("run_id", r.get("id", ""))) == rid for r in picked):
                    continue
                picked.append(row)

        return pd.DataFrame(picked)

    s_present = sample_with_repo_cap(present_df, n_present)
    s_absent  = sample_with_repo_cap(absent_df,  n_absent)
    out = pd.concat([s_present, s_absent], ignore_index=True)

    if len(out) < n_total:
        remaining = df.drop(columns=["_ttfts_present"], errors="ignore").copy()
        remaining = remaining.sample(frac=1.0, random_state=int(rng.integers(0, 2**31 - 1)))

        repo_counts = out[repo_col].astype(str).value_counts().to_dict()
        extra = []
        for _, row in remaining.iterrows():
            repo = str(row[repo_col])
            repo_counts.setdefault(repo, 0)
            if repo_counts[repo] >= max_per_repo:
                continue
            extra.append(row)
            repo_counts[repo] += 1
            if len(out) + len(extra) >= n_total:
                break

        if extra:
            out = pd.concat([out, pd.DataFrame(extra)], ignore_index=True)

    return out.drop(columns=["_ttfts_present"], errors="ignore")


# -------------------------
# MAIN
# -------------------------
def main() -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    rng = np.random.default_rng(SEED)

    ensure_dir(OUT_DIR)

    run_metrics_path = os.path.join(DATA_DIR, RUN_METRICS_FILE)
    if not os.path.exists(run_metrics_path):
        raise FileNotFoundError(f"Could not find Stage 3A file: {run_metrics_path}")

    if not os.path.exists(STAGE3C_PATH):
        raise FileNotFoundError(f"Could not find Stage 3C file: {STAGE3C_PATH}")

    # ---- Load Stage 3A (sampling frame)
    df = pd.read_csv(run_metrics_path, low_memory=False)

    run_id_col = pick_first_existing_col(df, ["run_id", "id", "workflow_run_id"])
    repo_col   = pick_first_existing_col(df, ["repo_full_name", "repo", "repository", "full_name"])
    style_col  = pick_first_existing_col(df, ["styles", "style", "execution_style", "exec_style", "style_label"])

    if run_id_col is None:
        raise ValueError("Could not find a run id column in 3A (tried: run_id/id/workflow_run_id).")
    if repo_col is None:
        raise ValueError("Could not find a repo column in 3A (tried: repo_full_name/repo/repository/full_name).")
    if style_col is None:
        raise ValueError("Could not find a styles column in 3A (tried: styles/style/execution_style/exec_style/style_label).")

    df[run_id_col] = normalize_run_id_series(df[run_id_col])

    df["_is_gmd"] = df[style_col].apply(lambda x: has_style(x, GMD_STYLE_TOKEN))
    df["_is_custom"] = df[style_col].apply(lambda x: has_style(x, CUSTOM_STYLE_TOKEN))

    gmd_df    = df[df["_is_gmd"]].copy()
    custom_df = df[(~df["_is_gmd"]) & (df["_is_custom"])].copy()
    other_df  = df[(~df["_is_gmd"]) & (~df["_is_custom"])].copy()

    # ---- Stratified sample
    s_gmd    = balanced_sample(gmd_df, N_GMD, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
    s_custom = balanced_sample(custom_df, N_CUSTOM, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
    s_other  = balanced_sample(other_df, N_OTHER, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)

    sample_df = pd.concat([s_gmd, s_custom, s_other], ignore_index=True)
    sample_df = sample_df.drop_duplicates(subset=[run_id_col]).copy()

    # Top-up if needed
    def top_up_from(pool: pd.DataFrame, needed: int, already: pd.DataFrame) -> pd.DataFrame:
        if needed <= 0:
            return already
        existing_ids = set(already[run_id_col].astype(str))
        pool2 = pool[~pool[run_id_col].astype(str).isin(existing_ids)].copy()
        add = balanced_sample(pool2, needed, TTFTS_PRESENT_RATIO, repo_col, rng, MAX_PER_REPO)
        return pd.concat([already, add], ignore_index=True).drop_duplicates(subset=[run_id_col])

    if len(sample_df) < TOTAL_N:
        sample_df = top_up_from(other_df,  TOTAL_N - len(sample_df), sample_df)
    if len(sample_df) < TOTAL_N:
        sample_df = top_up_from(custom_df, TOTAL_N - len(sample_df), sample_df)
    if len(sample_df) < TOTAL_N:
        sample_df = top_up_from(gmd_df,    TOTAL_N - len(sample_df), sample_df)

    if len(sample_df) > TOTAL_N:
        sample_df = sample_df.sample(n=TOTAL_N, random_state=SEED).copy()

    # ---- Only keep run_id and stratum
    sample_df["_stratum"] = np.where(
        sample_df[style_col].apply(lambda x: has_style(x, GMD_STYLE_TOKEN)), "GMD",
        np.where(sample_df[style_col].apply(lambda x: has_style(x, CUSTOM_STYLE_TOKEN)), "Custom", "Other")
    )
    sample_min = sample_df[[run_id_col, "_stratum"]].copy()
    sample_min.rename(columns={run_id_col: "run_id"}, inplace=True)
    sample_min["run_id"] = normalize_run_id_series(sample_min["run_id"])

    sampled_run_ids = set(sample_min["run_id"].astype(str))

    # ---- Load Stage 3C and filter to sampled run_ids
    df3c = pd.read_csv(STAGE3C_PATH, low_memory=False)

    run_id_col_3c = pick_first_existing_col(df3c, ["run_id", "workflow_run_id", "id"])
    if run_id_col_3c is None:
        raise ValueError("Could not find a run id column in 3C (tried: run_id/workflow_run_id/id).")

    df3c[run_id_col_3c] = normalize_run_id_series(df3c[run_id_col_3c])

    out_3c = df3c[df3c[run_id_col_3c].astype(str).isin(sampled_run_ids)].copy()

    # Normalize column name to "run_id" for cleanliness
    if run_id_col_3c != "run_id":
        out_3c.rename(columns={run_id_col_3c: "run_id"}, inplace=True)

    # ---- Add stratum (only extra column that is not originally from 3C)
    stratum_map = dict(zip(sample_min["run_id"].astype(str), sample_min["_stratum"]))
    out_3c["_stratum"] = out_3c["run_id"].astype(str).map(stratum_map)

    # Put _stratum first (optional), keep all other columns from 3C as-is
    cols = ["run_id", "_stratum"] + [c for c in out_3c.columns if c not in ("run_id", "_stratum")]
    out_3c = out_3c[cols]

    # ---- Save outputs
    out_csv  = os.path.join(OUT_DIR, OUT_CSV_NAME)
    out_txt  = os.path.join(OUT_DIR, "sampled_3c_all_rows_n68_run_ids.txt")
    out_meta = os.path.join(OUT_DIR, "sampled_3c_all_rows_n68_meta.json")

    out_3c.to_csv(out_csv, index=False)

    with open(out_txt, "w", encoding="utf-8") as f:
        for rid in sample_min["run_id"].astype(str).tolist():
            f.write(rid + "\n")

    meta = {
        "seed": SEED,
        "total_n_runs": TOTAL_N,
        "n_gmd": N_GMD,
        "n_custom": N_CUSTOM,
        "n_other": N_OTHER,
        "ttfts_present_ratio_target": TTFTS_PRESENT_RATIO,
        "max_per_repo": MAX_PER_REPO,
        "source_sampling_frame_3a": run_metrics_path,
        "source_stage3c": STAGE3C_PATH,
        "generated_at": datetime.now().isoformat(),
        "sampled_run_id_count": int(sample_min["run_id"].nunique()),
        "output_rows_in_3c": int(len(out_3c)),
        "rows_per_stratum_in_3c": out_3c["_stratum"].value_counts(dropna=False).to_dict(),
    }
    with open(out_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("[ok] Wrote:", out_csv)
    print("[ok] Wrote:", out_txt)
    print("[ok] Wrote:", out_meta)
    print("[ok] Sampled runs:", sample_min["run_id"].nunique(), " | Output 3C rows:", len(out_3c))


if __name__ == "__main__":
    if (N_GMD + N_CUSTOM + N_OTHER) != TOTAL_N:
        raise ValueError("N_GMD + N_CUSTOM + N_OTHER must equal TOTAL_N.")
    main()


[ok] Wrote: C:\Android Mobile App\ICST2026_Ext\0-Sampling_v2.0\sampled_3c_all_rows_n68.csv
[ok] Wrote: C:\Android Mobile App\ICST2026_Ext\0-Sampling_v2.0\sampled_3c_all_rows_n68_run_ids.txt
[ok] Wrote: C:\Android Mobile App\ICST2026_Ext\0-Sampling_v2.0\sampled_3c_all_rows_n68_meta.json
[ok] Sampled runs: 68  | Output 3C rows: 91


In [ ]:
##Ranodm Sample 68 runs from run_inventory